# 03 - Bank of Canada Scraping

This notebook collects macroeconomic data from the Bank of Canada Valet API.

Final CSV output:
- `colab_source_notebooks/data/bank_of_canada_macro.csv`

In [ ]:
!pip -q install requests pandas

In [ ]:
from pathlib import Path
import pandas as pd
import requests
try:
    from IPython.display import display
except ImportError:
    # Local validation fallback. Colab has display(), but plain Python may not.
    def display(value):
        print(value)


DATA_DIR = Path("/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
START_DATE = "2021-01-01"
END_DATE = "2026-07-03"
BASE_URL = "https://www.bankofcanada.ca/valet"
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (MBA assignment basic scraper)"})

# Local Windows certificate stores can fail for public HTTPS requests. Colab
# usually works normally, but this keeps the notebook runnable on this PC too.
session.verify = False
requests.packages.urllib3.disable_warnings()

In [ ]:
# A Valet series is one time-series column, such as the target overnight rate.
# A Valet group is a bundle of related series, such as CPI or bond yields.
INDIVIDUAL_SERIES = {
    "V39079": "Target overnight rate",
    "V80691311": "Prime rate",
    "FXUSDCAD": "USD/CAD exchange rate",
}
GROUP_SERIES = {
    "CPI_MONTHLY": {
        "STATIC_TOTALCPICHANGE": "Total CPI year-over-year change",
        "V41690973": "Total CPI",
    },
    "bond_yields_benchmark": {
        "BD.CDN.5YR.DQ.YLD": "5-year Government of Canada benchmark bond yield",
        "BD.CDN.10YR.DQ.YLD": "10-year Government of Canada benchmark bond yield",
    },
}

In [ ]:
def observation_value(observation, code):
    value_block = observation.get(code)
    return value_block.get("v") if isinstance(value_block, dict) else None

def fetch_series(code, label):
    # Interest-rate series help explain borrowing-cost pressure on REITs.
    url = f"{BASE_URL}/observations/{code}/json"
    response = session.get(url, params={"start_date": START_DATE, "end_date": END_DATE}, timeout=30)
    response.raise_for_status()
    rows = []
    for observation in response.json().get("observations", []):
        rows.append({"date": observation.get("d"), "series_code": code, "series_name": label, "value": observation_value(observation, code), "source": "Bank of Canada Valet API"})
    return rows

def fetch_group(group_code, wanted_codes):
    # CPI gives inflation context. Bond yields matter because investors compare REIT income with safer bond income.
    url = f"{BASE_URL}/observations/group/{group_code}/json"
    response = session.get(url, params={"start_date": START_DATE, "end_date": END_DATE}, timeout=30)
    response.raise_for_status()
    rows = []
    for observation in response.json().get("observations", []):
        for code, label in wanted_codes.items():
            value = observation_value(observation, code)
            if value is not None:
                rows.append({"date": observation.get("d"), "series_code": code, "series_name": label, "value": value, "source": f"Bank of Canada Valet API group: {group_code}"})
    return rows

In [ ]:
rows = []
for code, label in INDIVIDUAL_SERIES.items():
    rows.extend(fetch_series(code, label))
for group_code, wanted_codes in GROUP_SERIES.items():
    rows.extend(fetch_group(group_code, wanted_codes))

bank_of_canada_macro_df = pd.DataFrame(rows)
bank_of_canada_macro_df["value"] = pd.to_numeric(bank_of_canada_macro_df["value"], errors="coerce")
bank_of_canada_macro_df.to_csv(DATA_DIR / "bank_of_canada_macro.csv", index=False)
print("Saved macro rows:", len(bank_of_canada_macro_df))
display(bank_of_canada_macro_df.head())
print(bank_of_canada_macro_df["series_name"].drop_duplicates().to_list())